In [1]:
# Install Anthropic SDK
!pip install anthropic -q

import anthropic
import json
import time
import pandas as pd
from IPython.display import display, Markdown

# ── Set your API key ──────────────────────────────────────────
# In Colab: Runtime → Secrets → Add ANTHROPIC_API_KEY
# OR paste directly below (never commit to GitHub)
from google.colab import userdata
API_KEY = userdata.get('ANTHROPIC_API_KEY')
# If not using Colab secrets, replace with:
# API_KEY = "your-api-key-here"

client = anthropic.Anthropic(api_key=API_KEY)
MODEL = "claude-haiku-4-5-20251001"

# Claude Sonnet pricing (as of April 2026)
INPUT_COST_PER_1K  = 0.003   # $0.003 per 1K input tokens
OUTPUT_COST_PER_1K = 0.015   # $0.015 per 1K output tokens

print("✅ Setup complete. Model:", MODEL)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 17.2 MB/s eta 0:00:00
✅ Setup complete. Model: claude-haiku-4-5-20251001


In [8]:
TEST_CASES = [
    # Happy path — clear and specific
    "Serve Llama 3.1 70B at 500 requests per second, p95 latency under 200ms, budget $20K/month.",
    "Fine-tune Mistral 7B on 100K customer support transcripts. One-time job, need results in 48 hours.",

    # Edge cases — ambiguous or incomplete
    "We need to run some AI stuff for our e-commerce site. Maybe 1000 users a day?",
    "Training a large language model. Not sure of the size yet. Team of 5 ML engineers.",
    "Real-time fraud detection. Latency is critical. Processes 10M transactions per day.",

    # Complex — multi-constraint
    "Multi-modal model serving images and text. 200 RPS peak, 50 RPS average. Need to minimize cost but can't exceed 300ms latency. EU data residency required.",
    "RAG pipeline for enterprise search over 50M documents. Query volume unpredictable — spikes 10x during business hours.",

    # Adversarial — tests failure handling
    "I want the cheapest possible GPU setup. Don't care about latency or reliability.",
    "Need to train GPT-4 from scratch. Budget is $500.",

    # Specific hardware ask
    "Batch embedding job. 500M text chunks, needs to complete over a weekend. Which is better — H100 or A100?"
]

print(f"✅ {len(TEST_CASES)} test cases loaded")
for i, tc in enumerate(TEST_CASES):
    print(f"\n[{i+1}] {tc[:80]}..." if len(tc) > 80 else f"\n[{i+1}] {tc}")

#TEST_CASES = TEST_CASES[:5]  # 5 test cases instead of 10
#print(f"✅ Reduced to {len(TEST_CASES)} test cases for cost efficiency")


✅ 10 test cases loaded

[1] Serve Llama 3.1 70B at 500 requests per second, p95 latency under 200ms, budget ...

[2] Fine-tune Mistral 7B on 100K customer support transcripts. One-time job, need re...

[3] We need to run some AI stuff for our e-commerce site. Maybe 1000 users a day?

[4] Training a large language model. Not sure of the size yet. Team of 5 ML engineer...

[5] Real-time fraud detection. Latency is critical. Processes 10M transactions per d...

[6] Multi-modal model serving images and text. 200 RPS peak, 50 RPS average. Need to...

[7] RAG pipeline for enterprise search over 50M documents. Query volume unpredictabl...

[8] I want the cheapest possible GPU setup. Don't care about latency or reliability.

[9] Need to train GPT-4 from scratch. Budget is $500.

[10] Batch embedding job. 500M text chunks, needs to complete over a weekend. Which i...


In [9]:
STRATEGIES = {}

# ── Strategy 1: Zero-Shot ─────────────────────────────────────
# Bare instruction. No examples. No format. No reasoning guidance.
# Cheapest. Fastest. Baseline to beat.
STRATEGIES["1_zero_shot"] = {
    "name": "Zero-Shot",
    "system": """You are an AI infrastructure advisor.
Given a workload description, recommend the right GPU hardware configuration and estimated monthly cost.""",
    "user_template": "{workload}"
}

# ── Strategy 2: Few-Shot ──────────────────────────────────────
# 3 input→output examples added before the task.
# Model learns the pattern from examples rather than just instructions.
STRATEGIES["2_few_shot"] = {
    "name": "Few-Shot",
    "system": """You are an AI infrastructure advisor.
Given a workload description, recommend the right GPU hardware configuration and estimated monthly cost.

Here are three examples of good recommendations:

EXAMPLE 1:
Workload: Serve GPT-2 for a chatbot, 100 RPS, latency under 500ms, budget $5K/month.
Recommendation: 2x A100 40GB on Lambda Labs ($1.10/hr each = $1,584/mo).
Handles 100 RPS with headroom. Total: $1,584/mo — well within budget.

EXAMPLE 2:
Workload: One-time fine-tuning of Llama 7B on 10K examples. Need results in 24 hours.
Recommendation: 1x A100 80GB spot instance on CoreWeave ($1.80/hr).
Fine-tuning will complete in ~6 hours. Total cost: ~$11. Use spot to minimize cost.

EXAMPLE 3:
Workload: Batch inference on 10M records overnight. No latency requirement.
Recommendation: 4x A10G on Vast.ai spot ($0.35/hr each = $1.40/hr total).
Parallel batch processing completes in ~8 hours. Total: ~$11. Spot is safe for batch.""",
    "user_template": "Workload: {workload}\nRecommendation:"
}

# ── Strategy 3: Chain of Thought ─────────────────────────────
# Explicit reasoning instruction. Model thinks step by step before answering.
# Note: effective here because this IS a math/reasoning task (cost calculation).
STRATEGIES["3_chain_of_thought"] = {
    "name": "Chain of Thought",
    "system": """You are an AI infrastructure advisor.
Given a workload description, recommend the right GPU hardware configuration and estimated monthly cost.

Before giving your recommendation, think step by step:
1. What type of workload is this? (training / fine-tuning / inference / batch)
2. What are the hard constraints? (latency, throughput, budget, timeline)
3. What VRAM is required? (model size × precision)
4. Which GPU fits within VRAM requirements?
5. How many GPUs are needed to hit throughput targets?
6. What is the total monthly cost at current market rates?
7. Are there any risks or tradeoffs the user should know?

Show your reasoning for each step, then give your final recommendation.""",
    "user_template": "Workload: {workload}"
}

# ── Strategy 4: Structured Output ────────────────────────────
# XML template specifying exact required fields.
# Most reliable technique — forces consistent parseable output.
# Note: XML tags specifically boost Claude performance (trained on XML docs).
STRATEGIES["4_structured_output"] = {
    "name": "Structured Output",
    "system": """You are an AI infrastructure advisor.
Given a workload description, recommend the right GPU hardware configuration and estimated monthly cost.

ALWAYS respond using this exact XML structure. No other text before or after.

<recommendation>
  <workload_type>training | fine-tuning | inference | batch</workload_type>
  <hardware>
    <gpu_model>e.g. H100 SXM5 80GB</gpu_model>
    <gpu_count>number</gpu_count>
    <provider>e.g. Lambda Labs</provider>
    <hourly_rate>e.g. $2.49/hr per GPU</hourly_rate>
  </hardware>
  <monthly_cost>total monthly cost assuming 24/7</monthly_cost>
  <rationale>2-3 sentences explaining why this hardware fits the workload</rationale>
  <tradeoffs>key tradeoffs or risks to be aware of</tradeoffs>
  <confidence>high | medium | low</confidence>
</recommendation>""",
    "user_template": "Workload: {workload}"
}

# ── Strategy 5: Constitutional AI ────────────────────────────
# Model generates an answer, then critiques it against rules, then revises.
# Two-step: generate → self-check → revise if needed.
# Most expensive. Highest quality floor.
STRATEGIES["5_constitutional_ai"] = {
    "name": "Constitutional AI",
    "system": """You are an AI infrastructure advisor.
Given a workload description, recommend the right GPU hardware configuration and estimated monthly cost.

Follow this exact two-step process:

STEP 1 — DRAFT: Write your initial recommendation.

STEP 2 — SELF-CHECK: Review your draft against these rules:
  [ ] Did I specify a concrete GPU model (not vague like 'a powerful GPU')?
  [ ] Did I include a specific monthly cost estimate with math shown?
  [ ] Did I address all hard constraints mentioned in the workload?
  [ ] Did I flag any unrealistic expectations (e.g. budget too low)?
  [ ] Did I recommend spot vs reserved and explain why?

If any check fails, revise your recommendation before returning it.
Return only the final revised recommendation — do not show the self-check process.""",
    "user_template": "Workload: {workload}"
}

print("✅ Five strategies loaded:")
for k, v in STRATEGIES.items():
    tokens_estimate = len(v['system']) // 4
    print(f"  {v['name']:25s} — system prompt ~{tokens_estimate} tokens")

✅ Five strategies loaded:
  Zero-Shot                 — system prompt ~35 tokens
  Few-Shot                  — system prompt ~233 tokens
  Chain of Thought          — system prompt ~172 tokens
  Structured Output         — system prompt ~199 tokens
  Constitutional AI         — system prompt ~195 tokens


In [10]:
def calculate_cost(input_tokens, output_tokens):
    """Calculate actual API cost from token counts."""
    input_cost  = (input_tokens  / 1000) * INPUT_COST_PER_1K
    output_cost = (output_tokens / 1000) * OUTPUT_COST_PER_1K
    return round(input_cost + output_cost, 6)

def run_strategy(strategy_key, workload, test_case_num):
    """Run a single strategy against a single test case."""
    strategy = STRATEGIES[strategy_key]
    user_msg = strategy["user_template"].format(workload=workload)

    response = client.messages.create(
        model=MODEL,
        max_tokens=800,
        system=strategy["system"],
        messages=[{"role": "user", "content": user_msg}]
    )

    input_tokens  = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = calculate_cost(input_tokens, output_tokens)
    output_text = response.content[0].text

    return {
        "strategy": strategy["name"],
        "test_case": test_case_num,
        "workload": workload[:60] + "..." if len(workload) > 60 else workload,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "cost_usd": cost,
        "output": output_text,
        "quality_score": None  # you fill this in Cell 5
    }

# ── Run all 50 combinations ───────────────────────────────────
results = []
total_calls = len(STRATEGIES) * len(TEST_CASES)
call_num = 0

for strategy_key in STRATEGIES:
    print(f"\n{'='*60}")
    print(f"Running: {STRATEGIES[strategy_key]['name']}")
    print('='*60)

    for i, test_case in enumerate(TEST_CASES):
        call_num += 1
        print(f"  [{call_num}/{total_calls}] Test case {i+1}...", end=" ")

        try:
            result = run_strategy(strategy_key, test_case, i+1)
            results.append(result)
            print(f"✅ {result['input_tokens']}in + {result['output_tokens']}out = ${result['cost_usd']:.5f}")
        except Exception as e:
            print(f"❌ Error: {e}")

        time.sleep(0.5)  # rate limit buffer

print(f"\n✅ Experiment complete. {len(results)} results collected.")
print(f"Total cost: ${sum(r['cost_usd'] for r in results):.4f}")


Running: Zero-Shot
  [1/50] Test case 1... ✅ 70in + 800out = $0.01221
  [2/50] Test case 2... ✅ 64in + 572out = $0.00877
  [3/50] Test case 3... ✅ 56in + 389out = $0.00600
  [4/50] Test case 4... ✅ 54in + 424out = $0.00652
  [5/50] Test case 5... ✅ 54in + 762out = $0.01159
  [6/50] Test case 6... ✅ 76in + 800out = $0.01223
  [7/50] Test case 7... ✅ 61in + 800out = $0.01218
  [8/50] Test case 8... ✅ 51in + 394out = $0.00606
  [9/50] Test case 9... ✅ 49in + 400out = $0.00615
  [10/50] Test case 10... ✅ 62in + 408out = $0.00631

Running: Few-Shot
  [11/50] Test case 1... ✅ 362in + 531out = $0.00905
  [12/50] Test case 2... ✅ 356in + 372out = $0.00665
  [13/50] Test case 3... ✅ 348in + 364out = $0.00650
  [14/50] Test case 4... ✅ 346in + 395out = $0.00696
  [15/50] Test case 5... ✅ 346in + 419out = $0.00732
  [16/50] Test case 6... ✅ 368in + 621out = $0.01042
  [17/50] Test case 7... ✅ 353in + 449out = $0.00779
  [18/50] Test case 8... ✅ 343in + 365out = $0.00650
  [19/50] Test case 9... 

In [11]:
# ── Cell 5 (REVISED): LLM-as-Judge Auto-Scoring ──────────────
# Claude evaluates each output against the scoring rubric.
# Same pattern used in production eval pipelines (P3, P4).

JUDGE_SYSTEM_PROMPT = """You are an AI infrastructure expert evaluating
GPU hardware recommendations.

Score on a scale of 1-5:
5 - Specific GPU named, cost math shown, all constraints addressed, tradeoffs mentioned
4 - Specific GPU, cost estimate, most constraints addressed, minor gaps
3 - Correct direction but vague, missing cost math
2 - Partially relevant, missing key information
1 - Wrong, hallucinated, or unhelpful

You MUST respond with ONLY this JSON. No preamble. No explanation outside the JSON:
{"score": 4, "reason": "your reason here"}"""

import re

def judge_output(workload, recommendation):
    """Use Claude as judge to score a recommendation."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": f"Workload: {workload}\n\nRecommendation: {recommendation}"
        }]
    )

    raw = response.content[0].text.strip()

    # Try 1: direct JSON parse
    try:
        result = json.loads(raw)
        return result["score"], result["reason"]
    except:
        pass

    # Try 2: extract JSON block from response
    try:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            result = json.loads(match.group())
            return result["score"], result["reason"]
    except:
        pass

    # Try 3: extract just the score number
    try:
        score_match = re.search(r'"score"\s*:\s*([1-5])', raw)
        reason_match = re.search(r'"reason"\s*:\s*"([^"]+)"', raw)
        if score_match:
            score = int(score_match.group(1))
            reason = reason_match.group(1) if reason_match else "Score extracted"
            return score, reason
    except:
        pass

    # Fallback: print raw so you can debug
    print(f"\n  ⚠ Could not parse: {raw[:100]}")
    return 3, "Parse failed — defaulted to 3"
# ── Auto-score all 50 results ─────────────────────────────────
print("Running LLM-as-judge scoring on all 50 outputs...")
print("(This is the same eval pattern used in P3 and P4)\n")

for i, result in enumerate(results):
    print(f"[{i+1}/50] Scoring {result['strategy']} / TC{result['test_case']}...", end=" ")
    score, reason = judge_output(result["workload"], result["output"])
    result["quality_score"] = score
    result["judge_reason"] = reason
    print(f"Score: {score}/5 — {reason[:60]}...")
    time.sleep(0.3)  # rate limit buffer

print(f"\n✅ All 50 outputs scored automatically.")
print(f"Average score: {sum(r['quality_score'] for r in results)/len(results):.2f}/5")

Running LLM-as-judge scoring on all 50 outputs...
(This is the same eval pattern used in P3 and P4)

[1/50] Scoring Zero-Shot / TC1... Score: 5/5 — Score extracted...
[2/50] Scoring Zero-Shot / TC2... Score: 5/5 — Score extracted...
[3/50] Scoring Zero-Shot / TC3... Score: 2/5 — Score extracted...
[4/50] Scoring Zero-Shot / TC4... Score: 2/5 — Score extracted...
[5/50] Scoring Zero-Shot / TC5... Score: 5/5 — Score extracted...
[6/50] Scoring Zero-Shot / TC6... Score: 4/5 — Score extracted...
[7/50] Scoring Zero-Shot / TC7... Score: 4/5 — Score extracted...
[8/50] Scoring Zero-Shot / TC8... Score: 4/5 — Score extracted...
[9/50] Scoring Zero-Shot / TC9... Score: 5/5 — Exceptional response that directly addresses infeasibility w...
[10/50] Scoring Zero-Shot / TC10... Score: 4/5 — Strong recommendation with specific GPUs named (A100 vs H100...
[11/50] Scoring Few-Shot / TC1... Score: 5/5 — Score extracted...
[12/50] Scoring Few-Shot / TC2... Score: 5/5 — Specific GPU named (A100 40GB), sp

In [13]:
import warnings
warnings.filterwarnings('ignore')

# Build dataframe
df = pd.DataFrame(results)

# ── Per-strategy summary ──────────────────────────────────────
summary = df.groupby("strategy").agg(
    avg_input_tokens  = ("input_tokens",  "mean"),
    avg_output_tokens = ("output_tokens", "mean"),
    avg_cost_usd      = ("cost_usd",      "mean"),
    total_cost_usd    = ("cost_usd",      "sum"),
    avg_quality       = ("quality_score", "mean"),
).round(5)

# Quality per dollar — the core PM metric
# Higher = better quality for less money
summary["quality_per_dollar"] = (
    summary["avg_quality"] / (summary["avg_cost_usd"] * 1000)
).round(2)  # quality points per $0.001

# Extrapolate to production scale
DAILY_CALLS = 100_000
summary["daily_cost_100k_calls"] = (summary["avg_cost_usd"] * DAILY_CALLS).round(2)
summary["monthly_cost_100k_calls"] = (summary["daily_cost_100k_calls"] * 30).round(2)

# Sort by quality per dollar
summary = summary.sort_values("quality_per_dollar", ascending=False)

print("="*80)
print("PROMPT OPTIMIZATION RESULTS — Quality vs Cost Analysis")
print("="*80)
print()
print(summary[[
    "avg_input_tokens", "avg_output_tokens", "avg_cost_usd",
    "avg_quality", "quality_per_dollar", "daily_cost_100k_calls"
]].to_string())

print()
print("─"*80)
print("PRODUCTION COST PROJECTION (100K calls/day)")
print("─"*80)
for strategy, row in summary.iterrows():
    print(f"  {strategy:25s}  Daily: ${row['daily_cost_100k_calls']:>8,.2f}  "
          f"Monthly: ${row['monthly_cost_100k_calls']:>10,.2f}")

print()
best = summary.index[0]
worst = summary.index[-1]
print(f"✅ Best quality/dollar:  {best}")
print(f"⚠  Worst quality/dollar: {worst}")

PROMPT OPTIMIZATION RESULTS — Quality vs Cost Analysis

                   avg_input_tokens  avg_output_tokens  avg_cost_usd  avg_quality  quality_per_dollar  daily_cost_100k_calls
strategy                                                                                                                    
Structured Output             278.7              334.3       0.00585          3.9                0.67                  585.0
Few-Shot                      351.7              430.1       0.00751          4.1                0.55                  751.0
Zero-Shot                      59.7              574.9       0.00880          4.0                0.45                  880.0
Constitutional AI             225.7              572.2       0.00926          3.9                0.42                  926.0
Chain of Thought              203.7              790.3       0.01247          4.0                0.32                 1247.0

────────────────────────────────────────────────────────────────────

In [14]:
# ── Per test case breakdown ───────────────────────────────────
# See which strategies struggled on which types of inputs

print("QUALITY SCORES BY TEST CASE")
print("─"*80)

pivot = df.pivot_table(
    values="quality_score",
    index="test_case",
    columns="strategy",
    aggfunc="mean"
)

# Add test case labels
case_labels = [
    "TC1: Llama 70B 500 RPS (happy path)",
    "TC2: Fine-tune Mistral 7B (happy path)",
    "TC3: Vague e-commerce request (edge case)",
    "TC4: Unknown model size (edge case)",
    "TC5: Real-time fraud detection (edge case)",
    "TC6: Multi-modal multi-constraint (complex)",
    "TC7: RAG unpredictable traffic (complex)",
    "TC8: Cheapest possible (adversarial)",
    "TC9: Train GPT-4 for $500 (adversarial)",
    "TC10: H100 vs A100 batch job (specific ask)"
]

pivot.index = case_labels
print(pivot.to_string())

print()
print("─"*80)
print("AVERAGE QUALITY BY TEST CASE TYPE")
print("─"*80)
type_map = {
    1: "happy_path", 2: "happy_path",
    3: "edge_case",  4: "edge_case",  5: "edge_case",
    6: "complex",    7: "complex",
    8: "adversarial",9: "adversarial",
    10: "specific_ask"
}
df["case_type"] = df["test_case"].map(type_map)
type_summary = df.groupby(["strategy","case_type"])["quality_score"].mean().unstack().round(2)
print(type_summary.to_string())

QUALITY SCORES BY TEST CASE
────────────────────────────────────────────────────────────────────────────────
strategy                                     Chain of Thought  Constitutional AI  Few-Shot  Structured Output  Zero-Shot
TC1: Llama 70B 500 RPS (happy path)                       5.0                4.0       5.0                4.0        5.0
TC2: Fine-tune Mistral 7B (happy path)                    5.0                4.0       5.0                4.0        5.0
TC3: Vague e-commerce request (edge case)                 2.0                2.0       2.0                2.0        2.0
TC4: Unknown model size (edge case)                       4.0                4.0       3.0                4.0        2.0
TC5: Real-time fraud detection (edge case)                5.0                5.0       5.0                4.0        5.0
TC6: Multi-modal multi-constraint (complex)               4.0                4.0       5.0                4.0        4.0
TC7: RAG unpredictable traffic (complex)    

In [15]:
print("HILL CLIMB STRATEGY — Quality vs Cost Tradeoff")
print("="*80)
print()
print("Phase 1: Climb up for quality (ignore cost)")
print("Phase 2: Descend for cost (stop when quality degrades)")
print()

# Sort by quality descending — this is the quality climb order
climb = summary.sort_values("avg_quality", ascending=False)

print(f"{'Strategy':25s}  {'Avg Quality':12s}  {'Avg Cost':10s}  {'Daily @ 100K':15s}  {'Decision'}")
print("─"*85)

prev_quality = None
for strategy, row in climb.iterrows():
    q = row["avg_quality"]
    c = row["avg_cost_usd"]
    daily = row["daily_cost_100k_calls"]

    if prev_quality is None:
        decision = "← START HERE (best quality)"
    elif prev_quality - q < 0.3:
        decision = "✅ Acceptable quality drop — compress"
    elif prev_quality - q < 0.7:
        decision = "⚠  Moderate quality drop — test with users"
    else:
        decision = "❌ Significant quality drop — stop here"

    print(f"{strategy:25s}  {q:>12.2f}  ${c:>9.5f}  ${daily:>13,.2f}  {decision}")
    prev_quality = q

print()
print("─"*85)
print("PM Decision: What quality drop is acceptable for your product?")
print("  B2B enterprise: tolerate <0.2 quality drop")
print("  Consumer product: tolerate <0.5 quality drop")
print("  Internal tool: tolerate <1.0 quality drop")

HILL CLIMB STRATEGY — Quality vs Cost Tradeoff

Phase 1: Climb up for quality (ignore cost)
Phase 2: Descend for cost (stop when quality degrades)

Strategy                   Avg Quality   Avg Cost    Daily @ 100K     Decision
─────────────────────────────────────────────────────────────────────────────────────
Few-Shot                           4.10  $  0.00751  $       751.00  ← START HERE (best quality)
Chain of Thought                   4.00  $  0.01247  $     1,247.00  ✅ Acceptable quality drop — compress
Zero-Shot                          4.00  $  0.00880  $       880.00  ✅ Acceptable quality drop — compress
Structured Output                  3.90  $  0.00585  $       585.00  ✅ Acceptable quality drop — compress
Constitutional AI                  3.90  $  0.00926  $       926.00  ✅ Acceptable quality drop — compress

─────────────────────────────────────────────────────────────────────────────────────
PM Decision: What quality drop is acceptable for your product?
  B2B enterprise